# Speaker Authentication (SpeakerAuth): High-Level Overview

Before diving into the math, let's understand **what** we are computing and **why**.

### The Problem: Voice Verification
We want to unlock a device or authorize a user based on their voice. We have a stored "profile" of the user's voice, and we need to check if a new 1-second audio clip matches that profile.

### The Solution: The 2-Stage Pipeline
The workload consists of two distinct computational "kernels":

1.  **DSP Kernel (The "Ear"):** Raw audio is just air pressure vibrating over time. It's noisy and huge. We need to convert it into a compact digital fingerprint that represents the *timbre* of the voice (the shape of the user's throat and mouth), ignoring the pitch or volume. We call these fingerprints **MFCCs**.
2.  **ML Kernel (The "Brain"):** We take those fingerprints and compare them against a probabilistic model (GMM) of the user. The model calculates a "score" representing the probability that the audio came from this specific person.

### Computational Profile
* **Signal Processing (DSP):** Relies on **FFT** (Fast Fourier Transform) to analyze frequencies and **DCT** (Discrete Cosine Transform) to compress data.
* **Machine Learning (ML):** Relies on **Gaussian Probabilities**. This involves calculating distances between vectors and summing up probabilities using `exp()` and `log()` functions.

---

# Mathematical Walkthrough

This notebook provides a rigorous, step-by-step breakdown of the **Speaker Authentication** workload found in the EarBench suite.

**Objective:** Extract features from a raw audio signal $x[t]$ and compute the log-likelihood that it belongs to a target Speaker Model $\lambda$.

**Notation Key:**
* $t$: Time sample index (Raw Audio)
* $n$: Frame index (Time Steps)
* $k$: Frequency bin index (FFT)
* $m$: Mel-scale band index
* $c$: Cepstral coefficient index (MFCC)
* $j$: GMM Component index

In [ ]:
import numpy as np
import wave
import struct
import matplotlib.pyplot as plt

# Configuration (Matching EarBench / Librosa Defaults)
SAMPLE_RATE = 16000
N_MFCC = 13       # Number of coefficients to keep (c)
N_FFT = 512       # FFT size (k)
N_MELS = 26       # Number of Mel bands (m)
HOP_LENGTH = 512  # Stride between frames
WIN_LENGTH = 512  # Window size

## 1. Input & Padding

**Reasoning:** Spectral analysis assumes the signal is continuous. Abrupt edges (start/end of file) create artifacts called "spectral leakage." Padding ensures the first and last actual samples are analyzed in the center of a window, not the edge.
**Output:** `padded_audio` (1D Array of length $T + N_{FFT}$)
**Meaning:** The raw voltage signal extended with mirrored copies of itself at the boundaries.

---

Let the input signal be a discrete time series $x[t]$ of length $T$.

We add $P = \frac{N_{FFT}}{2}$ samples to both the start and end using **Reflect Padding**:

$$ x_{pad}[t] = \text{Reflect}(x[t], P) $$

In [ ]:
def pad_signal(audio, n_fft):
    pad_width = n_fft // 2
    return np.pad(audio, pad_width, mode='reflect')

# 1. Generate Dummy Input x[t] (1 second white noise)
np.random.seed(42)
dummy_audio = np.random.uniform(-0.5, 0.5, 16000)

# 2. Padding
padded_audio = pad_signal(dummy_audio, N_FFT)
print(f"Input x[t] length: {len(dummy_audio)}")
print(f"Padded length:     {len(padded_audio)}")

## 2. Framing & Windowing

**Reasoning:** Audio statistics change over time (non-stationary). To analyze frequency, we must chop the signal into short, stationary segments (frames). The Window function tapers the edges to zero to smooth the transition between frames.
**Output:** `frames` (2D Matrix: $N_{frames} \times N_{win}$)
**Meaning:** A collection of short "snapshots" of the audio, ready for independent processing.

---

We slice the signal into $N$ overlapping frames. 
For frame index $n$ and window sample index $i$ (where $0 \le i < N_{win}$):

$$ x_n[i] = x_{pad}[n \cdot H + i] \cdot w[i] $$

Where:
* $H$: Hop Length (`HOP_LENGTH`)
* $w[i]$: The **Hanning Window** function:

$$ w[i] = 0.5 \left( 1 - \cos \left( \frac{2\pi i}{N_{win} - 1} \right) \right) $$

In [ ]:
def frame_and_window(audio, win_length, hop_length):
    # Calculate number of frames N
    num_frames = 1 + int((len(audio) - win_length) / hop_length)
    
    # Vectorized Indexing
    indices = np.tile(np.arange(0, win_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * hop_length, hop_length), (win_length, 1)).T
    frames = audio[indices.astype(np.int32, copy=False)]
    
    # Apply Window w[i]
    frames *= np.hanning(win_length)
    return frames

frames = frame_and_window(padded_audio, WIN_LENGTH, HOP_LENGTH)
print(f"Windowed Frames x_n[i]: {frames.shape} (Frames n x Samples i)")

## 3. FFT & Power Spectrum

**Reasoning:** We need to know *what* frequencies are present in each frame, not just the raw waveform. The FFT converts Time $\to$ Frequency.
**Output:** `power_spec` (2D Matrix: $N_{frames} \times N_{FFT/2}$)
**Meaning:** A "Heatmap" showing how much energy exists at each frequency bin $k$ for each moment in time $n$.

---

We transform each frame $x_n$ using the **Discrete Fourier Transform (DFT)**.
For frequency bin $k$ (where $0 \le k \le \frac{N_{FFT}}{2} + 1$):

$$ X_n[k] = \sum_{i=0}^{N_{FFT}-1} x_n[i] \cdot e^{-j \frac{2\pi}{N_{FFT}} k i} $$

The **Power Spectrum** $P_n[k]$ is the normalized squared magnitude:

$$ P_n[k] = \frac{1}{N_{FFT}} |X_n[k]|^2 $$

In [ ]:
def compute_power_spectrum(frames, n_fft):
    # FFT (Real input)
    mag_frames = np.absolute(np.fft.rfft(frames, n_fft))
    
    # Power Spectrum P_n[k]
    pow_frames = ((1.0 / n_fft) * ((mag_frames) ** 2))
    return pow_frames

power_spec = compute_power_spectrum(frames, N_FFT)
print(f"Power Spectrum P_n[k]: {power_spec.shape} (Frames n x Bins k)")

## 4. Mel Filterbank Integration

**Reasoning:** The human ear cannot distinguish high frequencies well. We summarize the detailed FFT bins into broader "perceptual bands" (Mel bands). This reduces data size while keeping the information humans actually hear.
**Output:** `mel_spec` (2D Matrix: $N_{frames} \times N_{Mels}$)
**Meaning:** Energy distribution as perceived by a human ear (Logarithmic frequency scale).

---

We aggregate frequency bins $k$ into Mel-scale bands $m$. Let $\Phi_m[k]$ be the triangular weight of Mel band $m$ at frequency bin $k$.

**Slaney Normalization:** To ensure constant energy per band, we normalize the filter heights by their width in Hz:
$$ \Phi'_m[k] = \Phi_m[k] \cdot \frac{2}{f_{upper} - f_{lower}} $$

The Mel Spectrum $S_n[m]$ is computed via matrix multiplication:

$$ S_n[m] = \sum_{k} P_n[k] \cdot \Phi'_m[k] $$

In [ ]:
def hz_to_mel(freq):
    return 2595 * np.log10(1 + freq / 700.0)

def mel_to_hz(mel):
    return 700 * (10**(mel / 2595.0) - 1)

def build_mel_basis(sample_rate, n_fft, n_mels):
    # Calculate filter bank edges
    low_freq_mel = hz_to_mel(0)
    high_freq_mel = hz_to_mel(sample_rate / 2)
    mel_points = np.linspace(low_freq_mel, high_freq_mel, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)

    fbank = np.zeros((n_mels, int(n_fft / 2 + 1)))
    for m in range(1, n_mels + 1):
        f_m_minus = bin_points[m - 1]
        f_m = bin_points[m]
        f_m_plus = bin_points[m + 1]

        # Construct Triangles
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin_points[m - 1]) / (bin_points[m] - bin_points[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin_points[m + 1] - k) / (bin_points[m + 1] - bin_points[m])

    # Slaney Normalization (The 2/(upper-lower) factor)
    enorm = 2.0 / (hz_points[2:n_mels+2] - hz_points[:n_mels])
    fbank *= enorm[:, np.newaxis]
    return fbank

mel_basis = build_mel_basis(SAMPLE_RATE, N_FFT, N_MELS)

# Matrix Multiplication: (n x k) . (k x m)^T -> (n x m)
mel_spec = np.dot(power_spec, mel_basis.T)
print(f"Mel Spectrum S_n[m]: {mel_spec.shape} (Frames n x Mel Bands m)")

## 5. Log-Mel & Discrete Cosine Transform (DCT)

**Reasoning:** 
1. **Log:** Hearing is logarithmic (Decibels). We must match this scaling.
2. **DCT:** Mel bands are highly correlated (neighboring bands have similar energy). DCT decorrelates them, compressing the "shape" of the sound (timbre) into the first few coefficients, discarding noise.
**Output:** `mfcc` (2D Matrix: $N_{frames} \times N_{MFCC}$)
**Meaning:** The "Fingerprint" of the voice. These 13 numbers per frame describe *who* is speaking, independent of pitch or loudness.

---

1. **Log-Scaling (Decibels):**
   $$ L_n[m] = 10 \cdot \log_{10}(S_n[m]) $$
   *(Note: We add a small $\epsilon$ to avoid $\log(0)$)*

2. **DCT-II (Decorrelation):**
   We obtain the final MFCCs $c_n[i]$ by transforming the log-mel energies. For coefficient index $i$ ($0 \le i < N_{MFCC}$):
   
   $$ c_n[i] = \alpha_i \sum_{m=0}^{M-1} L_n[m] \cos \left[ \frac{\pi}{M} \left( m + \frac{1}{2} \right) i \right] $$
   
   Where $\alpha_i$ is the orthogonal normalization factor ($\sqrt{1/M}$ if $i=0$, $\sqrt{2/M}$ otherwise).

In [ ]:
# 1. Logarithm (dB)
log_mel_spec = 10 * np.log10(mel_spec + 1e-10)

# 2. DCT Basis Construction
def build_dct_basis(n_mfcc, n_mels):
    dct_matrix = np.zeros((n_mfcc, n_mels))
    for i in range(n_mfcc):
        # Ortho-Normalization Factor alpha
        if i == 0:
            alpha = np.sqrt(1.0 / n_mels)
        else:
            alpha = np.sqrt(2.0 / n_mels)
            
        for m in range(n_mels):
            dct_matrix[i, m] = alpha * np.cos(np.pi * i * (2 * m + 1) / (2 * n_mels))
    return dct_matrix

dct_basis = build_dct_basis(N_MFCC, N_MELS)

# Apply DCT: c_n[i] = Sum_m (L_n[m] * DCT[i, m])
mfcc = np.dot(log_mel_spec, dct_basis.T)
print(f"Final MFCCs c_n[i]: {mfcc.shape} (Frames n x Coeffs i)")

## 6. GMM Log-Likelihood Scoring

**Reasoning:** We have a "Dictionary" of 16 Gaussian shapes representing the user's voice. We calculate the probability that the incoming audio frame $c_n$ was generated by this dictionary.
**Output:** `final_score` (Scalar Float)
**Meaning:** The Total Log-Likelihood. A higher number (e.g., -5000 vs -20000) indicates the voice is a stronger match to the user.

---

We score the feature vectors $c_n$ against a Gaussian Mixture Model with $J$ components.
Parameters for component $j$: Weight $\pi_j$, Mean $\mu_j$, Diagonal Covariance $\Sigma_j$.

### Step A: Mahalanobis Distance
For frame $n$ and component $j$:
$$ D^2_{n,j} = \sum_{i=0}^{N_{MFCC}-1} \frac{(c_n[i] - \mu_j[i])^2}{\sigma^2_j[i]} $$

### Step B: Weighted Log-Probability
$$ \log \mathcal{P}(c_n | j) = \log \pi_j - \frac{1}{2} \left( D^2_{n,j} + N_{MFCC} \ln(2\pi) + \sum \ln(\sigma^2_j) \right) $$

### Step C: LogSumExp (Component Aggregation)
Total likelihood for frame $n$:
$$ \mathcal{L}_n = \ln \sum_{j=1}^{J} \exp( \log \mathcal{P}(c_n | j) ) $$

### Step D: Total Clip Score
$$ Score = \sum_{n=0}^{N_{frames}-1} \mathcal{L}_n $$

In [ ]:
def gmm_score(features, weights, means, covariances):
    n_samples, n_dim = features.shape
    n_components = len(weights)
    
    # Precompute constants
    log_weights = np.log(weights)
    log_2pi = np.log(2 * np.pi)
    
    # Matrix to store log P(c_n | j)
    log_prob = np.zeros((n_samples, n_components))
    
    for j in range(n_components):
        mu = means[j]
        sig = covariances[j]
        
        # A. Mahalanobis Distance (Squared)
        diff = features - mu
        mahalanobis = np.sum((diff ** 2) / sig, axis=1)
        
        # Log Determinant (sum of log variances)
        log_det = np.sum(np.log(sig))
        
        # B. Weighted Log-Probability
        log_prob[:, j] = log_weights[j] - 0.5 * (mahalanobis + n_dim * log_2pi + log_det)

    # C. LogSumExp Trick (Stable summation)
    # ln(sum(exp(x))) = max(x) + ln(sum(exp(x - max(x))))
    max_log = np.max(log_prob, axis=1, keepdims=True)
    frame_likelihoods = max_log + np.log(np.sum(np.exp(log_prob - max_log), axis=1, keepdims=True))
    
    # D. Total Score (Sum over n)
    return np.sum(frame_likelihoods)

# --- Test with Dummy Parameters ---
n_components = 16
# Random Model
dummy_weights = np.ones(n_components) / n_components
dummy_means = np.random.randn(n_components, N_MFCC)
dummy_covs = np.abs(np.random.randn(n_components, N_MFCC))

final_score = gmm_score(mfcc, dummy_weights, dummy_means, dummy_covs)
print(f"Final Log-Likelihood Score: {final_score:.4f}")